# EA3 — Procesamiento de datos en infraestructura Cloud


 **Dataset** | THE World University Rankings 2016–2026 (Kaggle) 



---
##  Diseño del esquema de datos

### Descripción del dataset

El dataset contiene los rankings universitarios mundiales publicados anualmente por **Times Higher Education (THE)** entre 2016 y 2026. Incluye métricas de calidad académica, investigación, impacto industrial y proyección internacional para cada universidad.

- **Filas:** 16,713
- **Columnas:** 14
- **Años cubiertos:** 2016–2026 (el ranking creció de 800 a 2,191 universidades)

---

### Diccionario de datos

| # | Campo | Tipo Spark | Nulable | Descripción |
|---|---|---|---|---|
| 1 | `Rank` | `DoubleType` | Sí | Posición en el ranking anual (valores enteros almacenados como float) |
| 2 | `Name` | `StringType` | No | Nombre oficial de la universidad |
| 3 | `Country` | `StringType` | No | País donde está ubicada la universidad |
| 4 | `Student Population` | `DoubleType` | Sí | Total de estudiantes matriculados |
| 5 | `Students to Staff Ratio` | `DoubleType` | Sí | Número de estudiantes por docente |
| 6 | `International Students` | `StringType` | Sí | Porcentaje de estudiantes internacionales (ej: `"26%"`) — 2 nulos |
| 7 | `Female to Male Ratio` | `StringType` | Sí | Proporción F:M (ej: `"46 : 54"`) — 760 nulos (4.5%) |
| 8 | `Overall Score` | `DoubleType` | No | Puntaje global THE (0–100) |
| 9 | `Teaching` | `DoubleType` | No | Puntaje en calidad de enseñanza |
| 10 | `Research Environment` | `DoubleType` | No | Puntaje en entorno de investigación |
| 11 | `Research Quality` | `DoubleType` | No | Puntaje en calidad de investigación |
| 12 | `Industry Impact` | `DoubleType` | No | Puntaje en impacto con la industria |
| 13 | `International Outlook` | `DoubleType` | No | Puntaje en proyección internacional |
| 14 | `Year` | `IntegerType` | No | Año de publicación del ranking |

---

### DDL equivalente en Spark SQL

```sql
CREATE TABLE IF NOT EXISTS university_rankings (
    Rank                      DOUBLE,
    Name                      STRING        NOT NULL,
    Country                   STRING        NOT NULL,
    `Student Population`      DOUBLE,
    `Students to Staff Ratio` DOUBLE,
    `International Students`  STRING,
    `Female to Male Ratio`    STRING,
    `Overall Score`           DOUBLE        NOT NULL,
    Teaching                  DOUBLE        NOT NULL,
    `Research Environment`    DOUBLE        NOT NULL,
    `Research Quality`        DOUBLE        NOT NULL,
    `Industry Impact`         DOUBLE        NOT NULL,
    `International Outlook`   DOUBLE        NOT NULL,
    Year                      INT           NOT NULL
)
USING delta;
```

---

### Diagrama del esquema

```
┌─────────────────────────────────────────────────────────┐
│               UNIVERSITY_RANKINGS (Delta)               │
├──────────────────────────┬────────────┬─────────────────┤
│ Campo                    │ Tipo       │ Nulable         │
├──────────────────────────┼────────────┼─────────────────┤
│ Rank                     │ DOUBLE     │ Sí              │
│ Name                     │ STRING     │ No  ← clave     │
│ Country                  │ STRING     │ No              │
│ Student Population       │ DOUBLE     │ Sí              │
│ Students to Staff Ratio  │ DOUBLE     │ Sí              │
│ International Students   │ STRING     │ Sí (2 nulos)    │
│ Female to Male Ratio     │ STRING     │ Sí (760 nulos)  │
│ Overall Score            │ DOUBLE     │ No              │
│ Teaching                 │ DOUBLE     │ No              │
│ Research Environment     │ DOUBLE     │ No              │
│ Research Quality         │ DOUBLE     │ No              │
│ Industry Impact          │ DOUBLE     │ No              │
│ International Outlook    │ DOUBLE     │ No              │
│ Year                     │ INT        │ No  ← clave     │
└──────────────────────────┴────────────┴─────────────────┘
  Clave natural compuesta: (Name, Year)
  Partición sugerida: Year
```

In [0]:
#  Versión de Spark y Python
import sys


print(f"  Versión de Spark  : {spark.version}")
print(f"  Versión de Python : {sys.version.split()[0]}")




  Versión de Spark  : 4.1.0
  Versión de Python : 3.11.10


In [0]:
#  Definición del esquema con StructType
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Rank",                    DoubleType(),  True),
    StructField("Name",                    StringType(),  False),
    StructField("Country",                 StringType(),  False),
    StructField("Student Population",      DoubleType(),  True),
    StructField("Students to Staff Ratio", DoubleType(),  True),
    StructField("International Students",  StringType(),  True),
    StructField("Female to Male Ratio",    StringType(),  True),
    StructField("Overall Score",           DoubleType(),  False),
    StructField("Teaching",                DoubleType(),  False),
    StructField("Research Environment",    DoubleType(),  False),
    StructField("Research Quality",        DoubleType(),  False),
    StructField("Industry Impact",         DoubleType(),  False),
    StructField("International Outlook",   DoubleType(),  False),
    StructField("Year",                    IntegerType(), False)
])

print("Esquema StructType definido correctamente:")
print(f"\n{'Campo':<30} {'Tipo':<15} {'Nulable'}")
print("-" * 55)
for field in schema.fields:
    print(f"{field.name:<30} {str(field.dataType):<15} {field.nullable}")

Esquema StructType definido correctamente:

Campo                          Tipo            Nulable
-------------------------------------------------------
Rank                           DoubleType()    True
Name                           StringType()    False
Country                        StringType()    False
Student Population             DoubleType()    True
Students to Staff Ratio        DoubleType()    True
International Students         StringType()    True
Female to Male Ratio           StringType()    True
Overall Score                  DoubleType()    False
Teaching                       DoubleType()    False
Research Environment           DoubleType()    False
Research Quality               DoubleType()    False
Industry Impact                DoubleType()    False
International Outlook          DoubleType()    False
Year                           IntegerType()   False


In [0]:
# lectura del dataset
FILE_PATH = "/Volumes/workspace/default/ea3_bigdata/THE World University Rankings 2016-2026.csv"

df = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("inferSchema", False)
    .schema(schema)
    .csv(FILE_PATH)
)

total_filas = df.count()
total_cols  = len(df.columns)

print(f"Dataset cargado exitosamente desde: {FILE_PATH}")
print(f"  Filas    : {total_filas:,}")
print(f"  Columnas : {total_cols}")
print()
print("Schema inferido por Spark:")
df.printSchema()

Dataset cargado exitosamente desde: /Volumes/workspace/default/ea3_bigdata/THE World University Rankings 2016-2026.csv
  Filas    : 16,713
  Columnas : 14

Schema inferido por Spark:
root
 |-- Rank: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Student Population: double (nullable = true)
 |-- Students to Staff Ratio: double (nullable = true)
 |-- International Students: string (nullable = true)
 |-- Female to Male Ratio: string (nullable = true)
 |-- Overall Score: double (nullable = true)
 |-- Teaching: double (nullable = true)
 |-- Research Environment: double (nullable = true)
 |-- Research Quality: double (nullable = true)
 |-- Industry Impact: double (nullable = true)
 |-- International Outlook: double (nullable = true)
 |-- Year: integer (nullable = true)



In [0]:
print("Primeras 10 filas del dataset:")
display(df.limit(10))

Primeras 10 filas del dataset:


Rank,Name,Country,Student Population,Students to Staff Ratio,International Students,Female to Male Ratio,Overall Score,Teaching,Research Environment,Research Quality,Industry Impact,International Outlook,Year
1.0,California Institute of Technology,United States,2243.0,6.9,26%,33 : 67,95.2,95.6,97.6,99.8,97.8,64.0,2016
2.0,University of Oxford,United Kingdom,19920.0,11.6,34%,46:54:00,94.2,86.5,98.9,98.8,73.1,94.4,2016
3.0,Stanford University,United States,15596.0,7.8,22%,42:58:00,93.9,92.5,96.2,99.9,63.3,76.3,2016
4.0,University of Cambridge,United Kingdom,18810.0,11.8,34%,46:54:00,92.8,88.2,96.7,97.0,55.0,91.5,2016
5.0,Massachusetts Institute of Technology,United States,11074.0,9.0,33%,37 : 63,92.0,89.4,88.6,99.7,95.4,84.0,2016
6.0,Harvard University,United States,20152.0,8.9,25%,null,91.6,83.6,99.0,99.8,45.2,77.2,2016
7.0,Princeton University,United States,7929.0,8.4,27%,45:55:00,90.1,85.1,91.9,99.3,52.1,78.5,2016
8.0,Imperial College London,United Kingdom,15060.0,11.7,51%,37 : 63,89.1,83.3,88.5,96.7,53.7,96.0,2016
9.0,ETH Zurich,Switzerland,18178.0,14.7,37%,31 : 69,88.3,77.0,95.0,91.1,80.0,97.9,2016
10.0,The University of Chicago,United States,14221.0,6.9,20%,42:58:00,87.9,85.7,88.9,99.2,36.6,65.0,2016


In [0]:
#  persistencia como tabla Delta en el metastore
TABLE_NAME = "university_rankings"

(
    df.write
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .saveAsTable(TABLE_NAME)
)

print(f"Tabla Delta '{TABLE_NAME}' creada exitosamente en el metastore de Databricks.")
print(f"Ubicación: dbfs:/user/hive/warehouse/{TABLE_NAME}")

Tabla Delta 'university_rankings' creada exitosamente en el metastore de Databricks.
Ubicación: dbfs:/user/hive/warehouse/university_rankings


In [0]:
# confirmación de la tabla creada con DESCRIBE TABLE
spark.sql("DESCRIBE TABLE university_rankings").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|Rank                   |double   |NULL   |
|Name                   |string   |NULL   |
|Country                |string   |NULL   |
|Student Population     |double   |NULL   |
|Students to Staff Ratio|double   |NULL   |
|International Students |string   |NULL   |
|Female to Male Ratio   |string   |NULL   |
|Overall Score          |double   |NULL   |
|Teaching               |double   |NULL   |
|Research Environment   |double   |NULL   |
|Research Quality       |double   |NULL   |
|Industry Impact        |double   |NULL   |
|International Outlook  |double   |NULL   |
|Year                   |int      |NULL   |
+-----------------------+---------+-------+



---
##  Validaciones en Spark y SQL

Cada validación se ejecuta **en paralelo** usando PySpark (API DataFrame) y Spark SQL, comparando sintaxis y resultados. Esto permite evidenciar las diferencias en expresividad y uso de cada enfoque.

In [0]:
#  SPARK: printSchema() — estructura del DataFrame
print("[SPARK] printSchema():")
df.printSchema()

[SPARK] printSchema():
root
 |-- Rank: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Student Population: double (nullable = true)
 |-- Students to Staff Ratio: double (nullable = true)
 |-- International Students: string (nullable = true)
 |-- Female to Male Ratio: string (nullable = true)
 |-- Overall Score: double (nullable = true)
 |-- Teaching: double (nullable = true)
 |-- Research Environment: double (nullable = true)
 |-- Research Quality: double (nullable = true)
 |-- Industry Impact: double (nullable = true)
 |-- International Outlook: double (nullable = true)
 |-- Year: integer (nullable = true)



In [0]:
# SQL: DESCRIBE TABLE — metadatos desde el metastore
print("[SQL] DESCRIBE TABLE university_rankings:")
spark.sql("DESCRIBE TABLE university_rankings").show(truncate=False)

[SQL] DESCRIBE TABLE university_rankings:
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|Rank                   |double   |NULL   |
|Name                   |string   |NULL   |
|Country                |string   |NULL   |
|Student Population     |double   |NULL   |
|Students to Staff Ratio|double   |NULL   |
|International Students |string   |NULL   |
|Female to Male Ratio   |string   |NULL   |
|Overall Score          |double   |NULL   |
|Teaching               |double   |NULL   |
|Research Environment   |double   |NULL   |
|Research Quality       |double   |NULL   |
|Industry Impact        |double   |NULL   |
|International Outlook  |double   |NULL   |
|Year                   |int      |NULL   |
+-----------------------+---------+-------+



In [0]:
# SQL: SHOW CREATE TABLE — DDL completo generado por Databricks
print("[SQL] SHOW CREATE TABLE university_rankings:")
spark.sql("SHOW CREATE TABLE university_rankings").show(truncate=False)

[SQL] SHOW CREATE TABLE university_rankings:
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt       

###  Descripción estadística de los datos

In [0]:
# SPARK: describe() — estadísticos descriptivos
print("[SPARK] df.describe() sobre columnas numéricas clave:")
df.describe(
    "Overall Score", "Teaching", "Research Quality",
    "Industry Impact", "International Outlook", "Student Population"
).show()

[SPARK] df.describe() sobre columnas numéricas clave:
+-------+------------------+------------------+------------------+------------------+---------------------+------------------+
|summary|     Overall Score|          Teaching|  Research Quality|   Industry Impact|International Outlook|Student Population|
+-------+------------------+------------------+------------------+------------------+---------------------+------------------+
|  count|             16713|             16713|             16713|             16713|                16713|             14529|
|   mean|   35.592513052013|28.593125112188158|49.909447735295956| 46.84396520235809|   48.121193083228526| 23138.19946314268|
| stddev|16.621830238490784|13.931517694085393|26.812393823640733|20.849751673355716|    22.61917858614752|33765.978439277016|
|    min| 8.222499999999998|               8.2|               0.7|               0.0|                  7.1|              25.0|
|    max|           98.4775|              99.2|          

In [0]:
# SQL: funciones agregadas equivalentes a describe()
print("[SQL] Estadísticos agregados equivalentes:")
spark.sql("""
    SELECT
        COUNT(*)                                 AS total_registros,
        ROUND(AVG(`Overall Score`),   2)         AS avg_overall_score,
        ROUND(MIN(`Overall Score`),   2)         AS min_overall_score,
        ROUND(MAX(`Overall Score`),   2)         AS max_overall_score,
        ROUND(STDDEV(`Overall Score`), 2)        AS stddev_overall_score,
        ROUND(AVG(`Teaching`),         2)        AS avg_teaching,
        ROUND(AVG(`Research Quality`), 2)        AS avg_research_quality
    FROM university_rankings
""").show()


[SQL] Estadísticos agregados equivalentes:
+---------------+-----------------+-----------------+-----------------+--------------------+------------+--------------------+
|total_registros|avg_overall_score|min_overall_score|max_overall_score|stddev_overall_score|avg_teaching|avg_research_quality|
+---------------+-----------------+-----------------+-----------------+--------------------+------------+--------------------+
|          16713|            35.59|             8.22|            98.48|               16.62|       28.59|               49.91|
+---------------+-----------------+-----------------+-----------------+--------------------+------------+--------------------+



**Interpretación:** Ambos métodos producen los mismos estadísticos. El `Overall Score` promedio es ~35.6 sobre 100, lo que refleja que la mayoría de universidades del ranking tienen puntajes bajos — las instituciones de élite (Oxford, MIT, Stanford) concentran los valores altos. La desviación estándar de ~16.6 confirma alta dispersión.

### Consultas SELECT con filtro

In [0]:
#SPARK: Top 10 universidades año 2026 por Overall Score
from pyspark.sql.functions import col, desc, asc

print("[SPARK] Top 10 universidades en 2026 por Overall Score:")
(
    df.filter(col("Year") == 2026)
    .select("Rank", "Name", "Country", "Overall Score")
    .orderBy(desc("Overall Score"))
    .limit(10)
    .show(truncate=False)
)

[SPARK] Top 10 universidades en 2026 por Overall Score:
+----+-------------------------------------+--------------+-----------------+
|Rank|Name                                 |Country       |Overall Score    |
+----+-------------------------------------+--------------+-----------------+
|1.0 |University of Oxford                 |United Kingdom|98.21            |
|2.0 |Massachusetts Institute of Technology|United States |97.67349999999999|
|3.0 |Princeton University                 |United States |97.211           |
|4.0 |University of Cambridge              |United Kingdom|97.20649999999999|
|5.0 |Stanford University                  |United States |97.151           |
|6.0 |Harvard University                   |United States |97.051           |
|7.0 |California Institute of Technology   |United States |96.31649999999999|
|8.0 |Imperial College London              |United Kingdom|94.689           |
|9.0 |University of California, Berkeley   |United States |94.406           |
|10.0|Ya

In [0]:
# SQL: equivalente al SELECT anterior
print("[SQL] Top 10 universidades en 2026 por Overall Score:")
spark.sql("""
    SELECT Rank, Name, Country, `Overall Score`
    FROM   university_rankings
    WHERE  Year = 2026
    ORDER BY `Overall Score` DESC
    LIMIT 10
""").show(truncate=False)

[SQL] Top 10 universidades en 2026 por Overall Score:
+----+-------------------------------------+--------------+-----------------+
|Rank|Name                                 |Country       |Overall Score    |
+----+-------------------------------------+--------------+-----------------+
|1.0 |University of Oxford                 |United Kingdom|98.21            |
|2.0 |Massachusetts Institute of Technology|United States |97.67349999999999|
|3.0 |Princeton University                 |United States |97.211           |
|4.0 |University of Cambridge              |United Kingdom|97.20649999999999|
|5.0 |Stanford University                  |United States |97.151           |
|6.0 |Harvard University                   |United States |97.051           |
|7.0 |California Institute of Technology   |United States |96.31649999999999|
|8.0 |Imperial College London              |United Kingdom|94.689           |
|9.0 |University of California, Berkeley   |United States |94.406           |
|10.0|Yale

###  Consultas GROUP BY

In [0]:
# SPARK: Top 10 países con más universidades en el ranking 2026
from pyspark.sql.functions import count

print("[SPARK] Top 10 países con más universidades en el ranking (2026):")
(
    df.filter(col("Year") == 2026)
    .groupBy("Country")
    .agg(count("*").alias("num_universidades"))
    .orderBy(desc("num_universidades"))
    .limit(10)
    .show(truncate=False)
)

[SPARK] Top 10 países con más universidades en el ranking (2026):
+------------------+-----------------+
|Country           |num_universidades|
+------------------+-----------------+
|United States     |171              |
|India             |128              |
|Japan             |115              |
|Turkey            |109              |
|United Kingdom    |109              |
|China             |97               |
|Iran              |90               |
|Russian Federation|80               |
|Brazil            |59               |
|Spain             |57               |
+------------------+-----------------+



In [0]:
# SQL: equivalente al GROUP BY anterior
print("[SQL] Top 10 países con más universidades en el ranking (2026):")
spark.sql("""
    SELECT   Country,
             COUNT(*) AS num_universidades
    FROM     university_rankings
    WHERE    Year = 2026
    GROUP BY Country
    ORDER BY num_universidades DESC
    LIMIT 10
""").show(truncate=False)

[SQL] Top 10 países con más universidades en el ranking (2026):
+------------------+-----------------+
|Country           |num_universidades|
+------------------+-----------------+
|United States     |171              |
|India             |128              |
|Japan             |115              |
|United Kingdom    |109              |
|Turkey            |109              |
|China             |97               |
|Iran              |90               |
|Russian Federation|80               |
|Brazil            |59               |
|Spain             |57               |
+------------------+-----------------+



In [0]:
# SPARK: Evolución del Overall Score promedio por año
from pyspark.sql.functions import avg, round as spark_round

print("[SPARK] Evolución del Overall Score promedio y total de universidades por año:")
(
    df.groupBy("Year")
    .agg(
        count("*").alias("total_universidades"),
        spark_round(avg("Overall Score"), 2).alias("avg_overall_score")
    )
    .orderBy(asc("Year"))
    .show()
)

[SPARK] Evolución del Overall Score promedio y total de universidades por año:
+----+-------------------+-----------------+
|Year|total_universidades|avg_overall_score|
+----+-------------------+-----------------+
|2016|                800|            38.17|
|2017|                981|            36.12|
|2018|               1103|            35.67|
|2019|               1258|            35.16|
|2020|               1397|            34.78|
|2021|               1526|            34.53|
|2022|               1662|            34.27|
|2023|               1799|            34.25|
|2024|               1904|            36.65|
|2025|               2092|            36.24|
|2026|               2191|            36.44|
+----+-------------------+-----------------+



In [0]:
#  SQL: equivalente a la evolución por año
print("[SQL] Evolución del Overall Score promedio y total de universidades por año:")
spark.sql("""
    SELECT   Year,
             COUNT(*)                         AS total_universidades,
             ROUND(AVG(`Overall Score`), 2)   AS avg_overall_score
    FROM     university_rankings
    GROUP BY Year
    ORDER BY Year ASC
""").show()

[SQL] Evolución del Overall Score promedio y total de universidades por año:
+----+-------------------+-----------------+
|Year|total_universidades|avg_overall_score|
+----+-------------------+-----------------+
|2016|                800|            38.17|
|2017|                981|            36.12|
|2018|               1103|            35.67|
|2019|               1258|            35.16|
|2020|               1397|            34.78|
|2021|               1526|            34.53|
|2022|               1662|            34.27|
|2023|               1799|            34.25|
|2024|               1904|            36.65|
|2025|               2092|            36.24|
|2026|               2191|            36.44|
+----+-------------------+-----------------+



###  Conteos, muestras y filtros adicionales

In [0]:
# SPARK: COUNT(*) total y nulos por columna
from pyspark.sql.functions import col, sum as spark_sum, when, isnan, isnull

total = df.count()
print(f"[SPARK] Total de registros: {total:,}\n")

print("[SPARK] Conteo de nulos por columna:")
null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show()

[SPARK] Total de registros: 16,713

[SPARK] Conteo de nulos por columna:
+----+----+-------+------------------+-----------------------+----------------------+--------------------+-------------+--------+--------------------+----------------+---------------+---------------------+----+
|Rank|Name|Country|Student Population|Students to Staff Ratio|International Students|Female to Male Ratio|Overall Score|Teaching|Research Environment|Research Quality|Industry Impact|International Outlook|Year|
+----+----+-------+------------------+-----------------------+----------------------+--------------------+-------------+--------+--------------------+----------------+---------------+---------------------+----+
|   0|   0|      0|              2184|                      0|                     2|                 760|            0|       0|                   0|               0|              0|                    0|   0|
+----+----+-------+------------------+-----------------------+---------------------

In [0]:
#  SQL: COUNT(*) y filtro con LIMIT
print("[SQL] Total de registros:")
spark.sql("SELECT COUNT(*) AS total_registros FROM university_rankings").show()

print("[SQL] Muestra de 5 registros con Overall Score > 80 en 2024:")
spark.sql("""
    SELECT Rank, Name, Country, `Overall Score`, Year
    FROM   university_rankings
    WHERE  Year = 2024
      AND  `Overall Score` > 80
    ORDER BY `Overall Score` DESC
    LIMIT 5
""").show(truncate=False)

[SQL] Total de registros:
+---------------+
|total_registros|
+---------------+
|          16713|
+---------------+

[SQL] Muestra de 5 registros con Overall Score > 80 en 2024:
+----+-------------------------------------+--------------+-----------------+----+
|Rank|Name                                 |Country       |Overall Score    |Year|
+----+-------------------------------------+--------------+-----------------+----+
|1.0 |University of Oxford                 |United Kingdom|98.45749999999998|2024|
|2.0 |Stanford University                  |United States |97.972           |2024|
|3.0 |Massachusetts Institute of Technology|United States |97.93            |2024|
|4.0 |Harvard University                   |United States |97.7905          |2024|
|5.0 |University of Cambridge              |United Kingdom|97.482           |2024|
+----+-------------------------------------+--------------+-----------------+----+



In [0]:
# SPARK: equivalente al filtro Overall Score > 80

print("[SPARK] Muestra de 5 registros con Overall Score > 80 en 2024:")
(
    df.filter((col("Year") == 2024) & (col("Overall Score") > 80))
    .select("Rank", "Name", "Country", "Overall Score", "Year")
    .orderBy(desc("Overall Score"))
    .limit(5)
    .show(truncate=False)
)

[SPARK] Muestra de 5 registros con Overall Score > 80 en 2024:
+----+-------------------------------------+--------------+-----------------+----+
|Rank|Name                                 |Country       |Overall Score    |Year|
+----+-------------------------------------+--------------+-----------------+----+
|1.0 |University of Oxford                 |United Kingdom|98.45749999999998|2024|
|2.0 |Stanford University                  |United States |97.972           |2024|
|3.0 |Massachusetts Institute of Technology|United States |97.93            |2024|
|4.0 |Harvard University                   |United States |97.7905          |2024|
|5.0 |University of Cambridge              |United Kingdom|97.482           |2024|
+----+-------------------------------------+--------------+-----------------+----+



### SQL vs Spark 

| Criterio | SQL | PySpark |
|---|---|---|
| Facilidad de uso | cualquiera que sepa SQL lo entiende de inmediato |  hay que aprender la API y el modelo de ejecución lazy |
| para consultas simples |  más legible y directo | más verboso, aunque funciona igual |
| Para transformaciones complejas | Se complica con CTEs anidadas |  se encadenan operaciones paso a paso |
| Escalabilidad | Depende del optimizador Catalyst | Control directo de particiones y caché |
| Integración con ML | No aplica directamente | Acceso nativo a MLlib y Streaming |
| Depuración | Difícil cuando el query falla | `df.explain()` muestra exactamente qué está pasando |

**Conclusión práctica:** para explorar datos rápido, SQL. 
Para construir pipelines o lógica que cambia según condiciones, PySpark. 
Lo ideal es combinar ambos en el mismo notebook.